In [ ]:
# imports
# built-in
import json

# local
from motiongen.data_handling import load_large_dataset, load_massive_prompts

# 3rd-party
import numpy as np
import pandas as pd
import scipy.stats as stats
import plotly.graph_objects as go

In [ ]:
# load data
prompts = load_massive_prompts()
clean, dirty = load_large_dataset("../data/large_dataset/filtering.csv")

In [ ]:
# print main prompts
labels = prompts[0:110:11]
for i, l in enumerate(labels):
    print(f"{i} - {l}")


In [ ]:
a = clean.groupby(by=["y"]).count().selected
b = dirty.groupby(by=["y"]).count().selected

In [ ]:
# proportion of  dirty samples per class
(b / (a+b)).sort_values(ascending=False)

In [ ]:
a = clean.groupby(by=["motion"]).count().selected
b = dirty.groupby(by=["motion"]).count().selected

In [ ]:
# proportion of dirty samples per prompt for 20 worst prompts
(b / (a+b)).sort_values(ascending=False)[:20]

In [ ]:
x = np.arange(1, 1000) / 1000
y = 0.1698 / x



fig = go.Figure(
    data=[
        go.Scatter(
            x=clean["naturalness"], y=clean["faithfulness"],
            marker={"color": "#777777", "symbol": "circle", "opacity": .1, "size": 10},
            mode="markers", name="accepted", showlegend=False
        ),
        go.Scatter(
            x=dirty["naturalness"], y=dirty["faithfulness"],
            marker={"color": "red", "symbol": "x", "opacity": 1, "line_width": 0, "size": 10},
            mode="markers", name="removed samples"
        ),
        go.Scatter(x=x, y=y, line={"color": "#777777", "dash": "dash"}, mode="lines", name="score=0.1698")
    ],
    layout={
        "height": 500, "width": 500,
        "margin": {'t': 5},
        "yaxis": {"title": "Faithfulness Score (MoBERT)", "scaleanchor": "x", "range": [0.21, .95], "gridcolor": "#DDDDDD"},
        "xaxis": {"title": "Naturalness Score (MoBERT)", "range": [0.21, 0.86], "gridcolor": "#DDDDDD"},
        "plot_bgcolor": "white",
        "font": {"size": 17},
        "legend": {
            "yanchor": "top",
            "y": 1.10,
            "xanchor": "left",
            "x": 0.01
        },
        "margin": {"l": 5, "t": 20, "r": 20, "b": 0},
    }
)

with open("distribution_removed_samples.svg", "wb") as fid:
    fid.write(fig.to_image("svg"))

fig

# Model training Results

In [ ]:
with open("results.json", "r") as fid:
    all_results = json.load(fid)

with open("class_results.json", "r") as fid:
    class_results = json.load(fid)["data"]

In [ ]:
def evaluate_results(run):
    results = all_results[run]
    results_mean = {"clean_test": {}, "poor_test": {}}
    results_mean["clean_test"]["clean"] =  np.mean(results["clean_test"]["clean"])
    results_mean["clean_test"]["infected"] =  np.mean(results["clean_test"]["infected"])
    results_mean["poor_test"]["clean"] =  np.mean(results["poor_test"]["clean"])
    results_mean["poor_test"]["infected"] =  np.mean(results["poor_test"]["infected"])
    display(pd.DataFrame(results_mean))

    results_std = {"clean_test": {}, "poor_test": {}}
    results_std["clean_test"]["clean"] =  np.std(results["clean_test"]["clean"])
    results_std["clean_test"]["infected"] =  np.std(results["clean_test"]["infected"])
    results_std["poor_test"]["clean"] =  np.std(results["poor_test"]["clean"])
    results_std["poor_test"]["infected"] =  np.std(results["poor_test"]["infected"])
    display(pd.DataFrame(results_std))

    display(stats.ttest_rel(results["clean_test"]["clean"], results["poor_test"]["clean"], alternative="greater"))
    display(stats.ttest_rel(results["clean_test"]["infected"], results["poor_test"]["infected"], alternative="greater"))

    display(stats.ttest_ind(results["clean_test"]["clean"], results["clean_test"]["infected"], alternative="greater"))

    display(np.mean(np.array(results["clean_test"]["infected"]) - np.array(results_mean["poor_test"]["infected"])))
    display(np.mean(np.array(results_mean["clean_test"]["clean"]) - np.array(results_mean["poor_test"]["clean"])))

In [ ]:
evaluate_results("long_run")

In [ ]:
for item in class_results:
    label = item["class"]
    good = np.array(item["good"])
    bad = np.array(item["bad"])
    test = stats.ttest_ind(good, bad, alternative="greater")
    print(f"Class {label} - {test}")

In [ ]:
for item in class_results:
    label = item["class"]
    good = np.array(item["good"])
    bad = np.array(item["bad"])
    test = stats.ttest_ind(good, bad, alternative="greater")
    print(f"Class {label} - {np.mean(good):.3f} +/- {np.std(good):.3f} | {np.mean(bad):.3f} +/- {np.std(bad):.3f}")